<a href="https://colab.research.google.com/github/XiayuZhou0524/AnalyticsEngineering2026/blob/main/2_03_practice_bike_hire_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# 2.03 Practice session
## Bike-hire trips

## How this practice works

This is a **short rehearsal** of the timed test. You do not need to finish or get the "right" answer. The aim is to practise the way of working.

- You have about **45 minutes**. Work individually.
- You may use **any AI tool** (for example Gemini in Colab). We are **not marking** anything today. In the real test, the quality of AI-generated code will not matter. What matters is the control youDshow.
- Write in markdown cells as you go: your thinking, **each prompt you send (paste it as you send it)**, and the checks you make.
- Unfinished work is fine. Say what you did not get to.
- At the end, download the Notebook (using File > Download > Download .ipynb). Once it has completed please zip the file up. This is the artefact you would submit for the in-class test.
- You can also save a copy to your GitHub using the instructions from my.wbs (1b asychronous tasks).

**Suggested timing**

| Minutes | Focus |
| --- | --- |
| 0 to 10 | Look at the tables and note what you find |
| 10 to 25 | Clean and prepare the data |
| 25 to 38 | Join and aggregate to answer the question |
| 38 to 45 | Check your answer and write up |

At the end, use **Runtime, then Restart session and run all** and note anything that fails.


## The data

A fictional bike-hire scheme: three tables covering trips taken in March 2025.

| Table | What it holds |
| --- | --- |
| `trips` | One record per trip: bike, start and end station, start time, duration and user type |
| `bikes` | The bikes, with type and purchase year |
| `stations` | The docking stations, with zone and capacity |

**How they connect:** `trips.bike_id` to `bikes.bike_id`, and `trips.start_station_id` (and `end_station_id`) to `stations.station_id`.

Check the real column names in the next cell before you write anything.


In [11]:
# ---- Setup: run this cell first ----
!pip install -q duckdb

import glob
import os

import duckdb
import pandas as pd

DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/dbsession/bike-hire-practice.zip"

DATA_DIR = "/content/bikes"
# Download only if no CSV files are there yet. The CSVs may sit inside a folder
# within the zip; the search below looks in sub-folders too.
if DATA_URL and not glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    import urllib.request
    import zipfile
    os.makedirs(DATA_DIR, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, "/content/bikes.zip")
    zipfile.ZipFile("/content/bikes.zip").extractall(DATA_DIR)

FILES = {"trips": "trips.csv", "bikes": "bikes.csv", "stations": "stations.csv"}

tables, missing = {}, []
for name, filename in FILES.items():
    found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    if found:
        tables[name] = pd.read_csv(found[0])
    else:
        missing.append(filename)

if missing:
    found_csvs = sorted(os.path.basename(f) for f in glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
    raise FileNotFoundError(
        f"Missing files: {missing}. CSV files found under {DATA_DIR}: {found_csvs}. Ask for help."
    )

# Make every table available as a DuckDB table, so you can use SQL or pandas.
con = duckdb.connect()
for name, df in tables.items():
    con.register(name, df)


def q(sql):
    """Run SQL against the tables above and return a pandas DataFrame."""
    return con.sql(sql).df()


for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print("   ", list(df.columns))

display(tables["trips"])
display(tables["bikes"])
display(tables["stations"])


trips: 612 rows, 7 columns
    ['trip_id', 'bike_id', 'start_station_id', 'end_station_id', 'start_time', 'duration_min', 'user_type']
bikes: 60 rows, 3 columns
    ['bike_id', 'bike_type', 'purchase_year']
stations: 12 rows, 4 columns
    ['station_id', 'station_name', 'zone', 'capacity']


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member
...,...,...,...,...,...,...,...
607,T0596,B051,S02,S02,2025-03-31 13:51:00,NaN,Casual
608,T0597,B011,S01,S04,2025-03-31 14:00:00,19.0,Member
609,T0598,B046,S06,S11,2025-03-31 15:38:00,12.0,Member
610,T0599,B027,S04,S02,31/03/2025 18:38,21.0,casual


,bike_id,bike_type,purchase_year
0,B001,Standard,2021
1,B002,Electric,2022
2,B003,electric,2024
3,B004,Standard,2021
4,B005,STANDARD,2022
5,B006,Standard,2024
6,B007,standard,2024
7,B008,Standard,2021
8,B009,Standard,2021
9,B010,Standard,2023


,station_id,station_name,zone,capacity
0,S01,Market Square,Centre,29.0
1,S02,Railway Station,Centre,23.0
2,S03,Canal Basin,North,24.0
3,S04,University Gate,West,28.0
4,S05,Park Road,South,22.0
5,S06,Library Corner,Centre,26.0
6,S07,Hospital Entrance,East,27.0
7,S08,Riverside Walk,South,14.0
8,S09,Old Town Hall,Centre,NaN
9,S10,Retail Park,North,16.0


In [14]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=tables["trips"])

https://docs.google.com/spreadsheets/d/1P5lGDXf_PJvZ1T-x0FwWEuP0OIsSNf1ZTsvXapjDaAk/edit#gid=0


In [16]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=tables["bikes"])

https://docs.google.com/spreadsheets/d/1RNRyq0sFtHdTXZF-9vvRKWTvdm7V6b6Cn8me7g1Teuk/edit#gid=0


In [17]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=tables["stations"])

https://docs.google.com/spreadsheets/d/1OhW0JlUytc02a834RfH77pXbJByopb-3BVOdlEuPTpY/edit#gid=0


## Examples: two ways to work with the tables

You can use SQL, pandas, or a mix of both. The next two cells show each. Run them and change them to see what happens. You do not have to use them in your answer.


In [2]:
# Example 1: SQL
# q() runs SQL against any of the tables above and returns a pandas DataFrame.
from IPython.display import display

display(q("SELECT * FROM trips LIMIT 5"))

display(q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT trip_id) AS n_distinct_trips FROM trips"))

display(q("""
    SELECT t.trip_id, s.station_name
    FROM trips AS t
    JOIN stations AS s ON t.start_station_id = s.station_id
    LIMIT 5
"""))


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member


,n_rows,n_distinct_trips
0,612,600


,trip_id,station_name
0,T0001,Park Road
1,T0002,Cathedral Close
2,T0003,Sports Centre
3,T0004,University Gate
4,T0005,Market Square


In [9]:
# Example 2: pandas
# Each table is also a pandas DataFrame, stored in the dictionary `tables`.
from IPython.display import display

trips = tables["trips"]
display(trips.head())
print(trips.shape)
print(trips.dtypes)

joined = tables["trips"].merge(tables["stations"], left_on="start_station_id", right_on="station_id", how="inner")
display(joined[["trip_id", "station_name"]].head())

# You can mix the two: run SQL, then use pandas methods on the result
result = q("SELECT start_station_id FROM trips")
print(result["start_station_id"].nunique(), "different start stations in trips")


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member


(612, 7)
trip_id              object
bike_id              object
start_station_id     object
end_station_id       object
start_time           object
duration_min        float64
user_type            object
dtype: object


,trip_id,station_name
0,T0001,Park Road
1,T0002,Cathedral Close
2,T0003,Sports Centre
3,T0004,University Gate
4,T0005,Market Square


12 different start stations in trips


## Your brief

**From:** the scheme manager

> We want to know whether electric bikes are used for longer trips than standard bikes, and whether that differs by zone. Please give us the **average trip duration by bike type and by the zone of the start station**.

![](https://drive.google.com/uc?export=view&id=1eH5Q3l4LnzwZ-XcT_kJDeHoKA2rufiOW)

**What your final table should look like**

One row per combination of bike type and zone, with the number of trips and the average duration in minutes. State what one row represents.

**Things to think about** (you do not have to solve all of them)

- Are there missing, duplicated or odd values in the tables you use? What will you do about each, and why?
- Do the bike types and dates look consistent?
- How will you check your final table is believable?


## 1. Look at the data

*What does one row of each table represent? Which columns will you use? What looks odd, missing or inconsistent? Write down what you find.*

#### 1.1 What does one row of each table represent?

### Table: `trips`

**Rows:** 612 | **Columns:** 7 | **One row =** one trip

| Column | Type | Description | Missing |
|---|---|---|---|
| `trip_id` | str | Unique trip identifier | 0 (0.0%) |
| `bike_id` | str | Bike used (FK to bikes.bike_id) | 0 (0.0%) |
| `start_station_id` | str | Station where trip started (FK to stations.station_id) | 0 (0.0%) |
| `end_station_id` | str | Station where trip ended (FK to stations.station_id) | 0 (0.0%) |
| `start_time` | str | Trip start timestamp (mixed formats in raw data) | 0 (0.0%) |
| `duration_min` | float64 | Trip duration in minutes | 30 (4.9%) |
| `user_type` | str | Member or Casual (inconsistent case in raw data) | 0 (0.0%) |

### Table: `bikes`

**Rows:** 60 | **Columns:** 3 | **One row =** one bike

| Column | Type | Description | Missing |
|---|---|---|---|
| `bike_id` | str | Unique bike identifier (PK) | 0 (0.0%) |
| `bike_type` | str | Standard or Electric (inconsistent spelling in raw data) | 0 (0.0%) |
| `purchase_year` | int64 | Year the bike was purchased | 0 (0.0%) |

### Table: `stations`

**Rows:** 12 | **Columns:** 4 | **One row =** one docking station

| Column | Type | Description | Missing |
|---|---|---|---|
| `station_id` | str | Unique station identifier (PK) | 0 (0.0%) |
| `station_name` | str | Station name | 0 (0.0%) |
| `zone` | str | Zone of the station (Centre/North/South/East/West) | 0 (0.0%) |
| `capacity` | float64 | Number of docking points | 1 (8.3%) |

#### 1.2 Which columns will you use?

| Table | Column | Used? | Why |
|---|---|---|---|
| `trips` | `trip_id` | Yes | Count trips; used to detect duplicated records |
| `trips` | `bike_id` | Yes | Join key to `bikes` |
| `trips` | `start_station_id` | Yes | Join key to `stations` (zone is based on the START station) |
| `trips` | `duration_min` | Yes | The measure to average (minutes) |
| `trips` | `start_time` | Yes | Check dates are consistent and all within March 2025 |
| `trips` | `end_station_id` | No | Brief asks for zone of the START station only |
| `trips` | `user_type` | No | Not needed for the question (cleaned only for consistency) |
| `bikes` | `bike_id` | Yes | Join key to `trips` |
| `bikes` | `bike_type` | Yes | Grouping variable (Standard vs Electric) |
| `bikes` | `purchase_year` | No | Not needed for the question |
| `stations` | `station_id` | Yes | Join key to `trips.start_station_id` |
| `stations` | `zone` | Yes | Grouping variable (zone of the start station) |
| `stations` | `station_name` | No | Zone is enough; name is only a label |
| `stations` | `capacity` | No | Not needed (and has 1 missing value, so ignored) |

#### 1.3 What looks odd, missing or inconsistent? Write down what you find.

## 1. Look at the data: what looks odd, missing or inconsistent?

| # | Table | Column | What I found | Rows affected | Why it matters for the brief |
|---|---|---|---|---|---|
| 1 | `trips` | all | Fully duplicated records (same `trip_id` and identical values) | 12 | Double-counts trips and distorts both the trip counts and the averages |
| 2 | `trips` | `duration_min` | Missing values | 30 (4.9%) | The average cannot be calculated for these trips |
| 3 | `trips` | `duration_min` | Extreme values: 1440, 1440 and 2880 minutes (24 to 48 hours). The 99th percentile is only about 52 minutes and the largest normal value is 90 | 3 | Likely unreturned bikes or data-entry errors; they would inflate the mean heavily |
| 4 | `trips` | `start_time` | Two date formats: `2025-03-01 05:52:00` and `01/03/2025 09:26` (day/month/year) | 98 | Stored as text, so dates cannot be reliably sorted or filtered until they are parsed |
| 5 | `trips` | `user_type` | Inconsistent case and trailing spaces: Member, member, MEMBER, Casual, CASUAL, casual, `Casual ` (7 spellings for 2 categories) | many | Not needed for this brief, but it would split groups if used |
| 6 | `bikes` | `bike_type` | Inconsistent spellings: Standard, STANDARD, standard, Electric, `Electric `, electric, E-bike (7 spellings for 2 types) | 60 bikes | **Critical**: this is the grouping variable, so grouping on it as-is would produce 7 groups instead of 2 |
| 7 | `stations` | `capacity` | One missing value (station S09) | 1 | Not used in this brief, so left as it is |

### Checks that were fine
- No orphan keys: every `bike_id` and `start_station_id` in `trips` exists in `bikes` and `stations`
- `bike_id` and `station_id` are unique in their own tables, so the joins will not multiply rows
- All dates fall within March 2025 once parsed
- No leading or trailing spaces in the join keys

### Assumptions I made
- `E-bike` means Electric
- `01/03/2025` means 1 March (day first), because every other trip is in March 2025
- Trips longer than 4 hours are errors, not real trips

In [10]:
for table_name, df in tables.items():
    print(f"\n{table_name}: {df.shape[0]} rows, {df.shape[1]} columns")
    print("Columns:", list(df.columns))
    display(df.head(5))
    display(df.dtypes)

for table_name, df in tables.items():
    print(f"\n--- {table_name} ---")
    print("Missing values:")
    display(df.isna().sum())

print("Duplicate trip IDs:",
      tables["trips"]["trip_id"].duplicated().sum())

print("Duplicate bike IDs:",
      tables["bikes"]["bike_id"].duplicated().sum())

print("Duplicate station IDs:",
      tables["stations"]["station_id"].duplicated().sum())

print("Bike types:")
display(tables["bikes"]["bike_type"].value_counts(dropna=False))

print("Trip duration summary:")
display(tables["trips"]["duration"].describe())


trips: 612 rows, 7 columns
Columns: ['trip_id', 'bike_id', 'start_station_id', 'end_station_id', 'start_time', 'duration_min', 'user_type']


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member


,0
trip_id,object
bike_id,object
start_station_id,object
end_station_id,object
start_time,object
duration_min,float64
user_type,object



bikes: 60 rows, 3 columns
Columns: ['bike_id', 'bike_type', 'purchase_year']


,bike_id,bike_type,purchase_year
0,B001,Standard,2021
1,B002,Electric,2022
2,B003,electric,2024
3,B004,Standard,2021
4,B005,STANDARD,2022


...

,0
station_id,0
station_name,0
zone,0
capacity,1


Duplicate trip IDs: 12
Duplicate bike IDs: 0
Duplicate station IDs: 0
Bike types:


,count
bike_type,
Standard,36
Electric,9
STANDARD,8
standard,3
Electric,2
electric,1
E-bike,1


Trip duration summary:


KeyError: 'duration'

## 2. Clean and prepare

*Decide what to do about each problem you found, say why, and do it. Short cells that you can re-run are better than one long cell.*

In [5]:
# Change these values if the printed column names are different.
DURATION_COL = "duration"
BIKE_TYPE_COL = "bike_type"
DURATION_UNIT = "minutes"  # Change to "seconds" if the source stores seconds.

trips_clean = tables["trips"].copy()
bikes_clean = tables["bikes"].copy()
stations_clean = tables["stations"].copy()

# Convert duration to a number.
trips_clean["duration_raw"] = pd.to_numeric(
    trips_clean[DURATION_COL],
    errors="coerce"
)

# Convert all durations to minutes.
if DURATION_UNIT == "seconds":
    trips_clean["duration_minutes"] = trips_clean["duration_raw"] / 60
elif DURATION_UNIT == "minutes":
    trips_clean["duration_minutes"] = trips_clean["duration_raw"]
else:
    raise ValueError("DURATION_UNIT must be 'seconds' or 'minutes'.")

# Parse trip start time.
trips_clean["start_time_clean"] = pd.to_datetime(
    trips_clean["start_time"],
    errors="coerce"
)

# Report duplicate trip IDs before removing them.
duplicate_trips = trips_clean[
    trips_clean.duplicated(subset="trip_id", keep=False)
]
print("Rows with duplicate trip IDs:", len(duplicate_trips))
display(duplicate_trips.head(10))

# Keep one row per trip ID.
# If duplicates contain conflicting values, investigate them before deciding.
trips_clean = trips_clean.drop_duplicates(
    subset="trip_id",
    keep="first"
)

# Standardize bike type text without changing the actual categories.
bikes_clean["bike_type_clean"] = (
    bikes_clean[BIKE_TYPE_COL]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Report invalid or missing durations.
invalid_duration = (
    trips_clean["duration_minutes"].isna()
    | (trips_clean["duration_minutes"] <= 0)
)
print("Rows with missing, zero, or negative duration:",
      int(invalid_duration.sum()))

# Remove rows that cannot contribute a valid positive trip duration.
trips_clean = trips_clean.loc[~invalid_duration].copy()

# The exercise data is for March 2025. Check dates before filtering.
print("Invalid start times:",
      int(trips_clean["start_time_clean"].isna().sum()))
print("Trip start-time range:",
      trips_clean["start_time_clean"].min(),
      "to",
      trips_clean["start_time_clean"].max())

march_2025 = (
    trips_clean["start_time_clean"].dt.to_period("M")
    == pd.Period("2025-03", freq="M")
)
print("Rows outside March 2025:", int((~march_2025).sum()))

# Keep March 2025 trips for this exercise.
trips_clean = trips_clean.loc[march_2025].copy()

KeyError: 'duration'

## 3. Join and aggregate

*Join the tables and calculate the answer. Note how many rows you expect after each join, then check.*

## 4. Check

*How do you know the answer is right? Compare a count, spot check a few records, or recalculate one number a different way. What did you find?*

## 5. Prompts and AI notes

*Paste each prompt as you sent it. Note what you accepted, changed or rejected and why.*

## 6. What I could not verify

*What remains unchecked? What would you do next?*